In [210]:
import pandas as pd
import numpy as np
from nhlpy import NHLClient

In [211]:
runner_raw = pd.read_csv("DATA/week_approach_maskedID_timeseries.csv")

In [212]:
runner_raw.head()

,nr. sessions,nr. rest days,total kms,max km one day,total km Z3-Z4-Z5-T1-T2,"nr. tough sessions (effort in Z5, T1 or T2)",nr. days with interval session,total km Z3-4,max km Z3-4 one day,total km Z5-T1-T2,...,max training success.2,avg recovery.2,min recovery.2,max recovery.2,Athlete ID,injury,rel total kms week 0_1,rel total kms week 0_2,rel total kms week 1_2,Date
0,5.0,2.0,22.2,16.4,11.8,1.0,2.0,10.0,10.0,0.6,...,0.0,0.18,0.16,0.20,0,0,0.718447,1.378882,1.919255,0
1,5.0,2.0,21.6,16.4,11.7,1.0,2.0,10.0,10.0,0.5,...,0.0,0.18,0.16,0.20,0,0,0.683544,1.018868,1.490566,1
2,5.0,2.0,21.6,16.4,11.7,1.0,2.0,10.0,10.0,0.5,...,0.0,0.17,0.16,0.18,0,0,0.683544,1.018868,1.490566,2
3,5.0,2.0,21.6,16.4,11.7,1.0,2.0,10.0,10.0,0.5,...,0.0,0.18,0.16,0.18,0,0,0.683544,1.018868,1.490566,3
4,6.0,1.0,39.2,17.6,18.9,1.0,3.0,17.2,10.0,0.5,...,0.0,0.17,0.16,0.18,0,0,2.202247,1.361111,0.618056,4


In [213]:
df = runner_raw.copy()
df = df.sort_values(by=['Athlete ID', 'Date']).reset_index(drop=True)
df.head()

,nr. sessions,nr. rest days,total kms,max km one day,total km Z3-Z4-Z5-T1-T2,"nr. tough sessions (effort in Z5, T1 or T2)",nr. days with interval session,total km Z3-4,max km Z3-4 one day,total km Z5-T1-T2,...,max training success.2,avg recovery.2,min recovery.2,max recovery.2,Athlete ID,injury,rel total kms week 0_1,rel total kms week 0_2,rel total kms week 1_2,Date
0,5.0,2.0,22.2,16.4,11.8,1.0,2.0,10.0,10.0,0.6,...,0.0,0.18,0.16,0.20,0,0,0.718447,1.378882,1.919255,0
1,5.0,2.0,21.6,16.4,11.7,1.0,2.0,10.0,10.0,0.5,...,0.0,0.18,0.16,0.20,0,0,0.683544,1.018868,1.490566,1
2,5.0,2.0,21.6,16.4,11.7,1.0,2.0,10.0,10.0,0.5,...,0.0,0.17,0.16,0.18,0,0,0.683544,1.018868,1.490566,2
3,5.0,2.0,21.6,16.4,11.7,1.0,2.0,10.0,10.0,0.5,...,0.0,0.18,0.16,0.18,0,0,0.683544,1.018868,1.490566,3
4,6.0,1.0,39.2,17.6,18.9,1.0,3.0,17.2,10.0,0.5,...,0.0,0.17,0.16,0.18,0,0,2.202247,1.361111,0.618056,4


In [214]:
names = []

for col in df.columns:
    if col == 'Athlete ID' or col == 'Date' or col == 'injury':
        pass
    else:
        names.append(col)

df.drop(columns=names, inplace=True)
df.head()

,Athlete ID,injury,Date
0,0,0,0
1,0,0,1
2,0,0,2
3,0,0,3
4,0,0,4


In [218]:
df['Injury_Date'] = df.apply(lambda row: row['Date'] if row['injury'] == 1 else np.nan, axis=1)

df['Last_Injury_Date'] = df.groupby('Athlete ID')['Injury_Date'].ffill()

df['Time_Since_Last_Injury'] = df['Date'] - df['Last_Injury_Date']

def assign_state_with_gaps(row):
    if row['injury'] == 1:
        return 'I'
    elif pd.isna(row['Time_Since_Last_Injury']):
        # If Time_Since_Last_Injury is NaN, they haven't been injured yet in the dataset
        return 'H'
    elif row['Time_Since_Last_Injury'] <= 31:
        # Not currently injured, but last injury was within the 31-day window
        return 'V'
    else:
        # Not currently injured, and it has been more than 31 days
        return 'H'

df['State'] = df.apply(assign_state_with_gaps, axis=1)

# Clean up the temporary calculation columns
df = df.drop(columns=['Injury_Date', 'Last_Injury_Date', 'Time_Since_Last_Injury'])

In [219]:
df['Next_State'] = df.groupby('Athlete ID')['State'].shift(-1)

transition_counts = pd.crosstab(df['State'], df['Next_State'])

# Normalize to get probabilities
transition_matrix = transition_counts.div(transition_counts.sum(axis=1), axis=0)

state_order = ['H', 'V', 'I']
transition_matrix = transition_matrix.reindex(index=state_order, columns=state_order).fillna(0)

print("Observation-to-Observation Transition Matrix P:")
print(transition_matrix)

Observation-to-Observation Transition Matrix P:
Next_State         H         V         I
State                                   
H           0.991305  0.000000  0.008695
V           0.114301  0.869685  0.016013
I           0.226691  0.433272  0.340037


In [220]:
import pandas as pd
from nhlpy import NHLClient
import time

client = NHLClient()

# Expand this list to your 30 target players. 
# Format: (Player_ID, Team_Abbr)
target_players = [
    (8478402, "EDM"), # Connor McDavid
    (8477492, "COL"), # Nathan MacKinnon
    (8478483, "TOR"), # Mitch Marner
    (8479318, "TOR"), # Auston Matthews
    (8477956, "BOS")  # David Pastrnak
]
season_str = "20232024"

all_players_data = []

for player_id, team_abbr in target_players:
    try:
        print(f"Processing Player {player_id}...")
        
        # 1. Fetch Team Schedule (to find game days)
        schedule_data = client.schedule.team_season_schedule(team_abbr=team_abbr, season=season_str)
        team_games = [g for g in schedule_data['games'] if g['gameType'] == 2]
        df_team = pd.DataFrame(team_games)[['gameDate']].rename(columns={'gameDate': 'Date'})
        df_team['Date'] = pd.to_datetime(df_team['Date'])
        df_team = df_team.sort_values('Date').reset_index(drop=True)
        
        # 2. Fetch Player Log (to find games actually played)
        # Note: using season_id as you discovered!
        log_data = client.stats.player_game_log(player_id=player_id, season_id=season_str, game_type=2)
        df_player = pd.DataFrame(log_data)[['gameDate', 'toi']].rename(columns={'gameDate': 'Date', 'toi': 'TOI'})
        df_player['Date'] = pd.to_datetime(df_player['Date'])
        
        # 3. Merge to find Missed Games
        df_games = pd.merge(df_team, df_player, on='Date', how='left')
        df_games['Is_Injured'] = df_games['TOI'].isna().astype(int)
        
        # 4. Create the Daily Calendar
        season_start = df_games['Date'].min()
        season_end = df_games['Date'].max()
        daily_calendar = pd.date_range(start=season_start, end=season_end, freq='D')
        df_daily = pd.DataFrame({'Date': daily_calendar})
        
        # 5. Merge game statuses onto the daily calendar
        df_daily = pd.merge(df_daily, df_games[['Date', 'Is_Injured']], on='Date', how='left')
        
        # Forward fill the Is_Injured status across rest days. 
        # If they start the season with a rest day before Game 1, bfill covers it.
        df_daily['Is_Injured'] = df_daily['Is_Injured'].ffill().bfill()
        
        # 6. Assign States (H, V, I) based on Daily logic
        # If Is_Injured was 1 anytime in the last 7 days, they are Vulnerable
        df_daily['Recent_Injuries'] = df_daily['Is_Injured'].shift(1).rolling(window=7, min_periods=1).max().fillna(0)
        
        def assign_daily_state(row):
            if row['Is_Injured'] == 1: 
                return 'I'
            elif row['Recent_Injuries'] > 0: 
                return 'V'
            else: 
                return 'H'
                
        df_daily['State'] = df_daily.apply(assign_daily_state, axis=1)
        
        # Tag and append
        df_daily['Player_ID'] = player_id
        all_players_data.append(df_daily[['Player_ID', 'Date', 'State']])
        
        # Polite delay to avoid API rate limiting (IP bans)
        time.sleep(1.5) 
        
    except Exception as e:
        print(f"Error processing player {player_id}: {e}")

# ---------------------------------------------------------
# Calculate the Market 3x3 Daily Transition Matrix P
# ---------------------------------------------------------
df_market = pd.concat(all_players_data, ignore_index=True)

# Shift strictly WITHIN each player's group so timelines don't bleed together
df_market['Next_State'] = df_market.groupby('Player_ID')['State'].shift(-1)

transition_counts = pd.crosstab(df_market['State'], df_market['Next_State'])
transition_matrix = transition_counts.div(transition_counts.sum(axis=1), axis=0)

state_order = ['H', 'V', 'I']
transition_matrix = transition_matrix.reindex(index=state_order, columns=state_order).fillna(0)

print("\nMarket Daily 3x3 Transition Matrix P:")
print(transition_matrix)

Processing Player 8478402...
Processing Player 8477492...
Processing Player 8478483...
Processing Player 8479318...
Processing Player 8477956...

Market Daily 3x3 Transition Matrix P:
Next_State         H         V         I
State                                   
H           0.994266  0.000000  0.005734
V           0.129032  0.838710  0.032258
I           0.000000  0.116279  0.883721
